# Instant-1.7B — Experiment 1: LFM3-1.7B with Thinking ON vs OFF

**Question:** how much accuracy does `LiquidAI/LFM3-1.7B` lose when its thinking step is turned off, and how many tokens (and seconds) does that save?

Nobody has published these numbers yet. This notebook measures them on seven standard benchmarks:

| Benchmark | Skill | Scored by | Liquid's score (thinking ON) |
|---|---|---|---|
| HumanEval+ | Python coding | passes all (extra-hard) tests — `evalplus` | – |
| MBPP+ | Python coding | passes all (extra-hard) tests — `evalplus` | – |
| LiveCodeBench v6 | harder, recent coding problems | passes all hidden tests | 73.5 |
| GSM8K | grade-school math | final number correct | – |
| IFEval | following instructions | Google's checker (strict) | 96.7 |
| IFBench | following new, unseen instructions | AllenAI's checker (strict) | 73.0 |
| BFCL (single-turn) | tool / function calling | official BFCL AST checker | 71.4 (full BFCL V4) |

Same model, same questions, same settings — the **only** difference is `enable_thinking=True` vs `False`.

**Kaggle setup (right sidebar → Session options):** Accelerator **GPU T4 x2**, Internet **On**. If you ran an older version of this notebook in the same session, use **Run → Factory reset** first.

**Built for speed (uses the whole Kaggle machine):**
- one worker process **per GPU** (both T4s run at the same time), 64 answers in parallel on each;
- answers are generated in chunks: when an answer finishes, a new question takes its place at once, so a GPU never waits idle for the single longest answer in its batch;
- scoring runs on the CPU **while** the GPUs are still generating, and the scorer self-test runs while the GPUs work;
- every finished answer is saved immediately: after a Kaggle disconnect, just *Run All* again and it continues.

Rough time with the default sizes (about 2,500 answers): install + load ≈ 10 min, then the experiment ≈ 1.5–2.5 h. The progress line shows a live estimate.

## 1. Install
Only light packages. We use plain `transformers` (like a normal fine-tuning notebook) — no vLLM, so Kaggle's PyTorch is never replaced.
Do **not** import torch before this cell.

In [ ]:
!pip install -q -U transformers accelerate
!pip install -q evalplus absl-py langdetect immutabledict nltk tabulate emoji "syllapy>=0.8.0"

## 2. Settings
Change these if you want. Bigger samples give tighter numbers (with 30 questions one question = 3.3 points; with 200 it is 0.5). To save time, lower the numbers in `N_PER_BENCH`.

In [ ]:
import os
os.environ["USE_TF"] = "0"           # we never use TensorFlow; avoids protobuf import errors on Kaggle
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL = "LiquidAI/LFM3-1.7B"
BENCHMARKS = ["humaneval", "mbpp", "livecodebench", "gsm8k", "ifeval", "ifbench", "bfcl"]
# Questions per benchmark (seeded random subset, the same for both modes). None = the whole benchmark.
N_PER_BENCH = {"humaneval": None,      # all 164
               "mbpp": 200,            # of 378
               "livecodebench": 50,    # easy + medium problems
               "gsm8k": 200,           # of 1,319
               "ifeval": 200,          # of 541
               "ifbench": 200}         # of 300
BFCL_PER_CATEGORY = 20    # BFCL: this many from each of its 7 single-turn categories (140 total)
MAX_NEW_TOKENS = 4096     # same cap for ON and OFF (fair). Answers that hit it count as "truncated"
LCB_MAX_NEW_TOKENS = 8192 # LiveCodeBench problems need longer thinking (still the same for both modes)
BATCH_SIZE = 64           # answers generated at the same time on EACH GPU (lowers itself on "out of memory")
SEED = 0

# Liquid's recommended sampling for LFM3 (from the model card)
SAMPLING = {
    "general": dict(temperature=0.1, top_k=50, top_p=1.0, repetition_penalty=1.1),   # code + instructions
    "math":    dict(temperature=1.0, top_k=20, top_p=0.95, repetition_penalty=1.0),  # GSM8K
}

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA_DIR = os.path.join(WORK, "data")
RESULTS_DIR = os.path.join(WORK, "results")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
print("Working dir:", WORK)

## 3. GPU check & load the model
T4 GPUs have no bfloat16, so we load in float16.

In [ ]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4 x2' in Session options."
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# T4 reports bf16 'supported' through slow emulation, so decide by hardware generation instead
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

# LFM3 ships its own model code, so trust_remote_code=True is required (as in Liquid's model card)
tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left", trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map={"": 0}, trust_remote_code=True)
model.eval()

# Tokens that end the assistant's turn
_eos = model.generation_config.eos_token_id
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])
if "<|im_end|>" in tokenizer.get_vocab():
    EOS_IDS.add(tokenizer.convert_tokens_to_ids("<|im_end|>"))
EOS_IDS.discard(None)

print(f"Loaded {MODEL} | {sum(p.numel() for p in model.parameters())/1e9:.2f}B params | {DTYPE} | "
      f"GPU memory used: {torch.cuda.memory_allocated(0)/1e9:.1f} GB | end tokens: {EOS_IDS}")

## 4. Template check — does the thinking switch really change the prompt?
With thinking OFF, LFM3's chat template should insert an empty `<think></think>`. If both prompts were identical, the whole experiment would be meaningless, so we stop here in that case.

In [ ]:
def render(messages, thinking, tools=None):
    # The model's own chat template; enable_thinking is the ON/OFF switch, tools go in for BFCL
    return tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=thinking)

def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False)) if text else 0

_msgs = [{"role": "user", "content": "What is 2+2?"}]
print("===== Thinking ON  =====\n" + render(_msgs, True))
print("===== Thinking OFF =====\n" + render(_msgs, False))
assert render(_msgs, True) != render(_msgs, False), "enable_thinking has no effect — stop and investigate"
print("OK: the two modes produce different prompts.")
with open(os.path.join(RESULTS_DIR, "template_check.txt"), "w") as f:
    f.write("ON:\n" + render(_msgs, True) + "\n\nOFF:\n" + render(_msgs, False))

## 5. Generation function (for single questions in this notebook)
Used for the quick looks, the speed test and the sandbox. The big benchmark run uses separate GPU workers (section 9) that work the same way but keep both GPUs full.

In [ ]:
_DROP = [t for t in tokenizer.all_special_tokens if t not in ("<think>", "</think>")]

def _clean(text):
    for t in _DROP:              # remove end-of-turn / padding markers, keep <think> tags
        text = text.replace(t, "")
    return text

@torch.no_grad()
def generate(prompts, sampling, max_new_tokens=MAX_NEW_TOKENS, batch_size=16, seed=SEED, label=""):
    torch.manual_seed(seed)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    results, n_batches = [None] * len(prompts), -(-len(prompts) // batch_size)
    do_sample = sampling["temperature"] > 0
    for b, start in enumerate(range(0, len(order), batch_size)):
        idx = order[start:start + batch_size]
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True,
                        add_special_tokens=False).to(model.device)
        t0 = time.perf_counter()
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                             temperature=sampling["temperature"] if do_sample else None,
                             top_k=sampling["top_k"] if do_sample else None,
                             top_p=sampling["top_p"] if do_sample else None,
                             repetition_penalty=sampling["repetition_penalty"],
                             pad_token_id=tokenizer.pad_token_id, eos_token_id=list(EOS_IDS))
        secs = time.perf_counter() - t0
        for row, i in zip(out[:, enc["input_ids"].shape[1]:].tolist(), idx):
            stop = next((k for k, t in enumerate(row) if t in EOS_IDS), None)
            toks = row if stop is None else row[:stop]
            results[i] = {"text": _clean(tokenizer.decode(toks, skip_special_tokens=False)),
                          "n_tokens": len(toks), "truncated": stop is None,
                          "seconds": secs / len(idx)}
        print(f"  {label} batch {b+1}/{n_batches}: {secs:.0f}s, longest answer "
              f"{max(results[i]['n_tokens'] for i in idx)} tokens", flush=True)
    return results

**Batching sanity check.** Padding short prompts in a batch must not change the answer. We generate one question greedily alone and inside a batch and compare. Tiny differences can come from float16 rounding; big ones mean you should set `BATCH_SIZE = 1` (slower but safe).

In [ ]:
_greedy = dict(temperature=0.0, top_k=None, top_p=None, repetition_penalty=1.0)
_q = render([{"role": "user", "content": "Name three primary colors."}], False)
_long = render([{"role": "user", "content": "Explain in detail how a hash map works, " * 20}], False)
_alone = generate([_q], _greedy, max_new_tokens=40, batch_size=1, label="alone")[0]["text"]
_batched = generate([_q, _long], _greedy, max_new_tokens=40, batch_size=2, label="batched")[0]["text"]
print("alone  :", repr(_alone)); print("batched:", repr(_batched))
_same = _alone.split()[:15] == _batched.split()[:15]
print("OK: batching does not change answers." if _same else
      "WARNING: answers differ inside a batch. Compare by eye; if they are clearly different, set BATCH_SIZE = 1.")

## 6. Helpers: separate the thinking from the answer, pull out code and numbers
We only grade the **final answer**. If thinking never finishes (hits the token cap), there is no answer, so it counts as wrong — we never grade half-finished reasoning.

In [ ]:
import re

def split_thinking(raw, prompt_opened_think=False):
    # returns (reasoning, answer, status) with status: closed / unclosed / none
    if "</think>" in raw:
        pre, post = raw.rsplit("</think>", 1)
        return pre.replace("<think>", "").strip(), post.strip(), "closed"
    if "<think>" in raw or prompt_opened_think:
        return raw.split("<think>", 1)[-1].strip(), "", "unclosed"
    return "", raw.strip(), "none"

def prompt_opens_thinking(prompt):
    # True if the chat template already wrote <think> and the model must close it
    tail = prompt.rstrip(); i = tail.rfind("<think>")
    return i != -1 and tail.find("</think>", i) == -1

def leaked_thinking(raw):
    # thinking OFF but the model wrote thinking tags anyway
    return "<think>" in raw or "</think>" in raw

_FENCE = re.compile(r"```[ \t]*([A-Za-z0-9_+-]*)[^\n]*\n(.*?)```", re.DOTALL)

def extract_code(answer, entry_point=None):
    blocks = [(lang.lower(), body) for lang, body in _FENCE.findall(answer)]
    if not blocks:  # no closed fence: take what follows an opening fence, else the whole answer
        m = re.search(r"```[ \t]*[A-Za-z0-9_+-]*[^\n]*\n(.*)", answer, re.DOTALL)
        return (m.group(1) if m else answer).strip("\n")
    py = [b for lang, b in blocks if lang in ("", "python", "py", "python3")] or [b for _, b in blocks]
    if entry_point:
        defining = [b for b in py if re.search(rf"\bdef\s+{re.escape(entry_point)}\s*\(", b)]
        if defining:
            return defining[-1].strip("\n")
    return max(py, key=len).strip("\n")

def leading_imports(code):
    return "\n".join(l for l in code.splitlines() if re.match(r"^(import|from)\s+\S+", l))

_NUM = re.compile(r"-?\d[\d,]*(?:\.\d+)?")

def _boxed(text):
    out, i = [], 0
    while (i := text.find("\\boxed{", i)) != -1:
        j = start = i + len("\\boxed{"); depth = 1
        while j < len(text) and depth:
            depth += {"{": 1, "}": -1}.get(text[j], 0); j += 1
        out.append(text[start:j - 1] if depth == 0 else text[start:]); i = j
    return out

def extract_final_number(text):
    # \boxed{} first, then '####', then 'answer is', then the last number
    boxed = _boxed(text)
    if boxed: cand = boxed[-1]
    elif "####" in text: cand = text.rsplit("####", 1)[-1]
    else:
        m = re.findall(r"answer\s*(?:is|:)\s*([^\n]+)", text, flags=re.IGNORECASE)
        cand = m[-1] if m else text
    nums = _NUM.findall(cand) or _NUM.findall(text)
    try: return float(nums[-1].replace(",", "")) if nums else None
    except ValueError: return None

# LFM3 tool calls (format from the model's tokenizer_config):
# <tool_call>\n<function=NAME>\n<parameter=KEY>\nVALUE\n</parameter>\n</function>\n</tool_call>
import ast as _ast
_CALL = re.compile(r"<tool_call>\s*<function=([^\n>]+)>(.*?)</function>\s*</tool_call>", re.DOTALL)
_PARAM = re.compile(r"<parameter=([^>\n]+)>\n?(.*?)\n?</parameter>", re.DOTALL)

def _typed(text, schema_type):
    # The template writes strings as-is and everything else as JSON; undo that using the schema type
    t = (schema_type or "").lower()
    if t in ("string", "any"):
        return text
    for parse in (json.loads, _ast.literal_eval):
        try:
            v = parse(text.strip())
            break
        except Exception:
            v = None
    else:
        return text
    if t == "float" and isinstance(v, int) and not isinstance(v, bool):
        v = float(v)
    return v

def parse_tool_calls(answer, functions=()):
    # -> [{name: {arg: value}}]  (the format BFCL's checker expects)
    props = {f["name"]: f.get("parameters", {}).get("properties", {}) for f in functions}
    calls = []
    for name, body in _CALL.findall(answer):
        name = name.strip()
        args = {k.strip(): _typed(v, props.get(name, {}).get(k.strip(), {}).get("type")) for k, v in _PARAM.findall(body)}
        calls.append({name: args})
    return calls

import json
_f = [{"name": "area", "parameters": {"properties": {"base": {"type": "integer"}, "unit": {"type": "string"}, "xs": {"type": "array"}}}}]
assert parse_tool_calls("<tool_call>\n<function=area>\n<parameter=base>\n10\n</parameter>\n<parameter=unit>\n10\n</parameter>\n"
                        "<parameter=xs>\n[1, 2]\n</parameter>\n</function>\n</tool_call>", _f) == [{"area": {"base": 10, "unit": "10", "xs": [1, 2]}}]
assert parse_tool_calls("no calls here", _f) == []

# quick self-checks of the helpers
assert split_thinking("<think>plan</think>\n\nAnswer: 4") == ("plan", "Answer: 4", "closed")
assert split_thinking("never closed", True)[1:] == ("", "unclosed")
assert prompt_opens_thinking("<|im_start|>assistant\n<think>\n")
assert not prompt_opens_thinking("<|im_start|>assistant\n<think>\n\n</think>\n\n")
assert extract_code("x\n```python\nx=1\n```\n```python\ndef f(a):\n    return a\n```", "f").startswith("def f")
assert extract_final_number("so \\boxed{1,234}") == 1234 and extract_final_number("#### 72") == 72
assert extract_final_number("The answer is 3.5 apples") == 3.5 and extract_final_number("none") is None
print("helpers OK")

# display panel used by the side-by-side views below
import html
from IPython.display import HTML, display

def _panel(title, color, g):
    think = (f"<details><summary>show thinking ({count_tokens(g['reasoning'])} tokens)</summary>"
             f"<pre style='white-space:pre-wrap;color:#555'>{html.escape(g['reasoning'])}</pre></details>") if g["reasoning"] else ""
    return (f"<div style='flex:1;background:#fff;color:#000;padding:12px;border-radius:10px;border-top:5px solid {color}'>"
            f"<b style='color:{color}'>{title}</b> — {g['n_tokens']} tokens, {g['seconds']:.1f}s"
            f"{' — <b>CUT OFF</b>' if g['truncated'] else ''}{think}"
            f"<pre style='white-space:pre-wrap'>{html.escape(g['answer'] or '(no answer)')}</pre></div>")

## 7. Load the benchmarks
Downloaded from their official sources. Each benchmark uses a **seeded random subset**, identical for both modes.

In [ ]:
import gzip, json, random, shutil, urllib.request

FILES = {
    "HumanEvalPlus.jsonl.gz": "https://github.com/evalplus/humanevalplus_release/releases/download/v0.1.10/HumanEvalPlus.jsonl.gz",
    "MbppPlus.jsonl.gz": "https://github.com/evalplus/mbppplus_release/releases/download/v0.2.0/MbppPlus.jsonl.gz",
    "gsm8k_test.jsonl": "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl",
    "ifeval_input_data.jsonl": "https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/data/input_data.jsonl",
}
# Pinned versions, so results are reproducible
IFBENCH_RAW = "https://raw.githubusercontent.com/allenai/IFBench/cd189a9c2889526d8af901d2103fd3e1ffcea6df"
BFCL_RAW = "https://raw.githubusercontent.com/ShishirPatil/gorilla/6ea57973c7a6097fd7c5915698c54c17c5b1b6c8/berkeley-function-call-leaderboard/bfcl_eval"
LCB_URL = "https://huggingface.co/datasets/livecodebench/code_generation_lite/resolve/0fe84c3912ea0c4d4a78037083943e8f0c4dd505/test6.jsonl"
BFCL_CATEGORIES = ["simple_python", "multiple", "parallel", "parallel_multiple", "live_simple", "live_multiple", "irrelevance"]

FILES["ifbench_test.jsonl"] = f"{IFBENCH_RAW}/data/IFBench_test.jsonl"
FILES["ifbench_sample_output.jsonl"] = f"{IFBENCH_RAW}/data/sample_output.jsonl"          # for the scorer self-test
FILES["ifbench_official_strict.jsonl"] = f"{IFBENCH_RAW}/eval/eval_results_strict.jsonl"  # official scores of those outputs
for c in BFCL_CATEGORIES:
    FILES[f"BFCL_v4_{c}.json"] = f"{BFCL_RAW}/data/BFCL_v4_{c}.json"
    if c != "irrelevance":   # irrelevance has no answer key: the right answer is "no tool call"
        FILES[f"BFCL_v4_{c}_answer.json"] = f"{BFCL_RAW}/data/possible_answer/BFCL_v4_{c}.json"
FILES["livecodebench_v6.jsonl"] = LCB_URL   # ~130 MB

def download(url, dest, tries=4):
    for k in range(tries):
        try:
            req = url
            if k == tries - 1 and "/releases/download/" in url:   # last try: same file via the GitHub API
                owner, repo, _, _, tag, name = url.split("github.com/")[1].split("/")
                api = json.load(urllib.request.urlopen(f"https://api.github.com/repos/{owner}/{repo}/releases/tags/{tag}", timeout=60))
                asset = next(a["url"] for a in api["assets"] if a["name"] == name)
                req = urllib.request.Request(asset, headers={"Accept": "application/octet-stream"})
            with urllib.request.urlopen(req, timeout=120) as r, open(dest + ".part", "wb") as f:
                shutil.copyfileobj(r, f)
            os.replace(dest + ".part", dest); return
        except Exception as e:
            print(f"  download problem ({e}), retrying..."); time.sleep(3 * (k + 1))
    raise RuntimeError(f"could not download {url}")

for name, url in FILES.items():
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print("downloading", name); download(url, path)

def read_jsonl(name):
    path = os.path.join(DATA_DIR, name)
    with (gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, encoding="utf-8")) as f:
        return [json.loads(l) for l in f if l.strip()]

def subset(rows, n, seed=SEED):
    if not n or n >= len(rows): return rows
    keep = set(random.Random(seed).sample(range(len(rows)), n))
    return [r for i, r in enumerate(rows) if i in keep]

CODE_INSTRUCTION = ("Return the complete function implementation, including the signature and any "
                    "imports it needs, in a single ```python code block.")
HE_ROWS, MBPP_ROWS = read_jsonl("HumanEvalPlus.jsonl.gz"), read_jsonl("MbppPlus.jsonl.gz")

def load_items(bench):
    n, items = N_PER_BENCH.get(bench), []
    if bench == "humaneval":
        for r in subset(HE_ROWS, n):
            q = f"Complete the following Python function.\n\n```python\n{r['prompt'].rstrip()}\n```\n\n{CODE_INSTRUCTION}"
            items.append(dict(bench=bench, id=r["task_id"], question=q, entry_point=r["entry_point"], prompt=r["prompt"]))
    elif bench == "mbpp":
        for r in subset(MBPP_ROWS, n):
            task = r["prompt"].strip().strip('"').strip()
            q = f"{task}\n\nThe function must be named `{r['entry_point']}` and pass the test above.\n\n{CODE_INSTRUCTION}"
            items.append(dict(bench=bench, id=r["task_id"], question=q, entry_point=r["entry_point"]))
    elif bench == "gsm8k":
        rows = [dict(bench=bench, id=f"gsm8k/{i}", gold=float(r["answer"].rsplit("####", 1)[-1].strip().replace(",", "")),
                     question=f"{r['question']}\n\nSolve the problem and put the final numeric answer inside \\boxed{{}}.")
                for i, r in enumerate(read_jsonl("gsm8k_test.jsonl"))]
        items = subset(rows, n)
    elif bench == "ifeval":
        items = subset([dict(bench=bench, id=f"ifeval/{r['key']}", question=r["prompt"], row=r)
                        for r in read_jsonl("ifeval_input_data.jsonl")], n)
    elif bench == "ifbench":
        items = subset([dict(bench=bench, id=f"ifbench/{r['key']}", question=r["prompt"], row=r)
                        for r in read_jsonl("ifbench_test.jsonl")], n)
    elif bench == "bfcl":
        for c in BFCL_CATEGORIES:
            answers = ({} if c == "irrelevance" else
                       {a["id"]: a["ground_truth"] for a in read_jsonl(f"BFCL_v4_{c}_answer.json")})
            for r in subset(read_jsonl(f"BFCL_v4_{c}.json"), BFCL_PER_CATEGORY):
                msgs = r["question"][0]
                items.append(dict(bench=bench, id=r["id"], category=c, messages=msgs,
                                  question=[m["content"] for m in msgs if m["role"] == "user"][-1],
                                  functions=r["function"], tools=bfcl_tools(r["function"]),
                                  ground_truth=answers.get(r["id"])))
    elif bench == "livecodebench":
        rows = [r for r in LCB_ROWS if r["difficulty"] in ("easy", "medium")]
        for r in subset(rows, n):
            items.append(dict(bench=bench, id=f"lcb/{r['question_id']}", question=lcb_question(r),
                              messages=[{"role": "system", "content": LCB_SYSTEM}, {"role": "user", "content": lcb_question(r)}],
                              difficulty=r["difficulty"], row=r))
    for it in items:   # plain one-message chat unless the benchmark set its own messages
        it.setdefault("messages", [{"role": "user", "content": it["question"]}])
    return items

def bfcl_tools(functions):
    # BFCL uses Python-style type names; the chat template expects JSON-Schema ones (as BFCL's own OSS handlers do)
    fix = {"dict": "object", "float": "number", "tuple": "array", "any": "string"}
    def conv(x):
        if isinstance(x, dict):
            return {k: (fix.get(v, v) if k == "type" and isinstance(v, str) else conv(v)) for k, v in x.items()}
        return [conv(v) for v in x] if isinstance(x, list) else x
    return [{"type": "function", "function": conv(f)} for f in functions]

# LiveCodeBench: official prompt wording; hidden tests are stored zlib+pickle — decoded with a
# restricted unpickler that refuses any code or objects (only plain text can load)
import base64, io, pickle, zlib
LCB_SYSTEM = ("You are an expert Python programmer. You will be given a question (problem specification) and "
              "will generate a correct Python program that matches the specification and passes all tests.")

def lcb_question(r):
    q = f"### Question:\n{r['question_content']}\n\n"
    if r["starter_code"]:
        q += ("### Format: You will use the following starter code to write the solution to the problem and "
              f"enclose your code within delimiters.\n```python\n{r['starter_code']}\n```\n\n")
    else:
        q += ("### Format: Read the inputs from stdin solve the problem and write the answer to the regular output "
              "(stdout). Do not directly test on the sample inputs. Enclose your code within delimiters as follows. "
              "Ensure that when the python program runs, it reads the inputs, runs the algorithm and writes output "
              "to STDOUT.\n```python\n# YOUR CODE HERE\n```\n\n")
    return q + "### Answer: (use the provided format with backticks)\n\n"

class _NoCodeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError(f"blocked {module}.{name}")

def lcb_tests(r):
    priv = json.loads(_NoCodeUnpickler(io.BytesIO(zlib.decompress(base64.b64decode(r["private_test_cases"])))).load())
    return json.loads(r["public_test_cases"]) + priv

LCB_ROWS = read_jsonl("livecodebench_v6.jsonl")

ITEMS = {b: load_items(b) for b in BENCHMARKS}
SAMPLING_OF = {"humaneval": "general", "mbpp": "general", "livecodebench": "general", "gsm8k": "math",
               "ifeval": "general", "ifbench": "general", "bfcl": "general"}
MAX_TOKENS_OF = {b: (LCB_MAX_NEW_TOKENS if b == "livecodebench" else MAX_NEW_TOKENS) for b in BENCHMARKS}
for b, its in ITEMS.items():
    print(f"{b:14s} {len(its):4d} problems | example: {its[0]['question'][:80]!r}")

## 8. Scoring
- **Code:** the official `evalplus` checker runs each answer against the original tests plus many extra hard tests (`plus_pass`).
- **Math:** final number must match the reference.
- **Instructions:** Google's original IFEval checker.

In [ ]:
import subprocess, sys, nltk
for p in ("punkt", "punkt_tab"):
    nltk.download(p, quiet=True)

# Google's IFEval checker code (downloaded once)
IFEVAL_DIR = os.path.join(WORK, "vendor", "instruction_following_eval")
os.makedirs(IFEVAL_DIR, exist_ok=True)
for f in ["instructions.py", "instructions_registry.py", "instructions_util.py", "evaluation_lib.py"]:
    if not os.path.exists(os.path.join(IFEVAL_DIR, f)):
        download(f"https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/{f}",
                 os.path.join(IFEVAL_DIR, f))
open(os.path.join(IFEVAL_DIR, "__init__.py"), "a").close()
sys.path.insert(0, os.path.dirname(IFEVAL_DIR))
from instruction_following_eval import evaluation_lib as ifeval_lib

def code_solution(it, answer):
    code = extract_code(answer, it["entry_point"])
    if it["bench"] == "humaneval":
        if f"def {it['entry_point']}" not in code:      # model wrote only the body
            return it["prompt"] + code
        return leading_imports(it["prompt"]) + "\n\n" + code
    return code

def score_code(bench, items, answers, tag):
    # evalplus needs every problem in the file; the ones we did not pick get an empty solution and are ignored
    rows, fname, env_key = ((HE_ROWS, "HumanEvalPlus.jsonl.gz", "HUMANEVAL_OVERRIDE_PATH") if bench == "humaneval"
                            else (MBPP_ROWS, "MbppPlus.jsonl.gz", "MBPP_OVERRIDE_PATH"))
    work = os.path.join(RESULTS_DIR, "scoring", tag); os.makedirs(work, exist_ok=True)
    samples = os.path.join(work, f"{bench}_samples.jsonl")
    sol = {it["id"]: code_solution(it, a) for it, a in zip(items, answers)}
    with open(samples, "w") as f:
        for r in rows:
            f.write(json.dumps({"task_id": r["task_id"], "solution": sol.get(r["task_id"], "")}) + "\n")
    res_file = samples.replace(".jsonl", "_eval_results.json")
    if os.path.exists(res_file): os.remove(res_file)      # evalplus would silently reuse old results
    env = {**os.environ, env_key: os.path.join(DATA_DIR, fname)}
    # Generous time limits (same for both modes): on a busy machine evalplus's defaults can time out
    # even the official reference solution on its huge extra tests.
    p = subprocess.run([sys.executable, "-m", "evalplus.evaluate", bench, "--samples", samples,
                        "--parallel", str(max(1, (os.cpu_count() or 2) // 2)),
                        "--min-time-limit", "4", "--gt-time-limit-factor", "8"],
                       env=env, capture_output=True, text=True)
    if p.returncode != 0 or not os.path.exists(res_file):
        raise RuntimeError("evalplus failed:\n" + p.stdout[-2000:] + p.stderr[-2000:])
    ev = json.load(open(res_file))["eval"]
    out = []
    for it in items:
        r = ev[it["id"]][0]; base = r["base_status"] == "pass"
        out.append({"correct": base and r["plus_status"] == "pass", "base_pass": base})
    return out

# ---- IFBench: AllenAI's checker files (pinned), loaded as a local package
import importlib.util
IFB_DIR = os.path.join(WORK, "vendor", "ifbench")
os.makedirs(IFB_DIR, exist_ok=True)
for f in ["__init__.py", "classic_instructions.py", "instructions.py", "instructions_registry.py", "instructions_util.py"]:
    if not os.path.exists(os.path.join(IFB_DIR, f)):
        download(f"{IFBENCH_RAW}/ifbench/{f}", os.path.join(IFB_DIR, f))
if not os.path.exists(os.path.join(WORK, "vendor", "ifbench_evaluation_lib.py")):
    download(f"{IFBENCH_RAW}/evaluation_lib.py", os.path.join(WORK, "vendor", "ifbench_evaluation_lib.py"))
for p in ("stopwords", "averaged_perceptron_tagger_eng"):
    nltk.download(p, quiet=True)
def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod); return mod
ifbench_lib = _load_module("ifbench_evaluation_lib", os.path.join(WORK, "vendor", "ifbench_evaluation_lib.py"))

# ---- BFCL: the official AST checker (pinned). Its imports pull in the whole bfcl_eval package
# (30+ heavy dependencies), so we swap them for tiny stand-ins; we only check Python functions.
_bfcl_path = os.path.join(WORK, "vendor", "bfcl_ast_checker.py")
if not os.path.exists(_bfcl_path):
    download(f"{BFCL_RAW}/eval_checker/ast_eval/ast_checker.py", _bfcl_path + ".orig")
    src = open(_bfcl_path + ".orig").read()
    src = re.sub(r"^from bfcl_eval[^\n]*\([^)]*\)\n|^from bfcl_eval[^\n]*\n", "", src, flags=re.M)
    stub = ('import enum\n'
            'class Language(enum.Enum):\n    PYTHON = "python"\n    JAVA = "java"\n    JAVASCRIPT = "javascript"\n'
            'class _Cfg:\n    underscore_to_dot = False\n'
            'class _AnyModel(dict):\n    def __getitem__(self, k):\n        return _Cfg\n'
            'MODEL_CONFIG_MAPPING = _AnyModel()\nJAVA_TYPE_CONVERSION = {}\nJS_TYPE_CONVERSION = {}\n'
            'def java_type_converter(*a, **k):\n    raise NotImplementedError\njs_type_converter = java_type_converter\n')
    open(_bfcl_path, "w").write(stub + src)
bfcl_checker = _load_module("bfcl_ast_checker", _bfcl_path)

def score_bfcl(items, answers):
    out = []
    for it, a in zip(items, answers):
        calls = parse_tool_calls(a, it["functions"])
        if it["category"] == "irrelevance":          # right answer: do NOT call any tool
            out.append({"correct": len(calls) == 0, "n_calls": len(calls)}); continue
        if not calls:
            out.append({"correct": False, "n_calls": 0}); continue
        try:
            res = bfcl_checker.ast_checker(it["functions"], calls, it["ground_truth"],
                                           bfcl_checker.Language.PYTHON, it["category"], "lfm3")
            ok = bool(res["valid"])
        except Exception as e:                        # malformed call = wrong
            ok = False
        out.append({"correct": ok, "n_calls": len(calls)})
    return out

# ---- LiveCodeBench: run the program on every public + hidden test (stop at the first failure)
from concurrent.futures import ThreadPoolExecutor
import tempfile
LCB_TIMEOUT = 6   # seconds per test, as in LiveCodeBench

_FUNC_HARNESS = '''
import sys, json
sys.setrecursionlimit(1 << 25)
from typing import *
import collections, heapq, bisect, itertools, functools, math, string, re, random, copy, operator
from collections import *
from heapq import *
from bisect import *
from itertools import *
from functools import *
from math import *
@@CODE@@
_args = [json.loads(l) for l in sys.stdin.read().split("\\n") if l.strip()]
print(json.dumps(Solution().@@FUNC@@(*_args)))
'''

def _limits():
    try:
        import resource
        resource.setrlimit(resource.RLIMIT_AS, (4 << 30, 4 << 30))   # 4 GB memory cap per test
    except Exception:
        pass

def _same(got, expected, functional):
    if functional:
        try:
            g, e = json.loads(got.strip().splitlines()[-1]), json.loads(expected)
        except Exception:
            return False
        if isinstance(g, float) or isinstance(e, float):
            try: return abs(float(g) - float(e)) <= 1e-6 * max(1, abs(float(e)))
            except Exception: return False
        return g == e
    gl, el = [l.strip() for l in got.strip().splitlines()], [l.strip() for l in expected.strip().splitlines()]
    if gl == el: return True
    if len(gl) != len(el): return False
    for a, b in zip(gl, el):
        ta, tb = a.split(), b.split()
        if len(ta) != len(tb): return False
        for x, y in zip(ta, tb):
            if x == y: continue
            try:
                if abs(float(x) - float(y)) > 1e-6 * max(1, abs(float(y))): return False
            except ValueError:
                return False
    return True

def lcb_run(row, code):
    func = json.loads(row["metadata"] or "{}").get("func_name")
    prog = _FUNC_HARNESS.replace("@@CODE@@", code).replace("@@FUNC@@", func) if func else code
    tests = lcb_tests(row)
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "solution.py"); open(path, "w").write(prog)
        for k, t in enumerate(tests):
            try:
                p = subprocess.run([sys.executable, path], input=t["input"], capture_output=True, text=True,
                                   timeout=LCB_TIMEOUT, cwd=d, preexec_fn=_limits if os.name == "posix" else None)
            except subprocess.TimeoutExpired:
                return {"correct": False, "passed": k, "n_tests": len(tests), "fail": "timeout"}
            if p.returncode != 0 or not _same(p.stdout, t["output"], bool(func)):
                return {"correct": False, "passed": k, "n_tests": len(tests), "fail": "error" if p.returncode else "wrong"}
    return {"correct": True, "passed": len(tests), "n_tests": len(tests), "fail": ""}

def score_lcb(items, answers):
    def one(pair):
        it, a = pair
        code = extract_code(a) if a.strip() else ""
        return lcb_run(it["row"], code) if code.strip() else {"correct": False, "passed": 0, "n_tests": 0, "fail": "no code"}
    with ThreadPoolExecutor(max_workers=max(1, os.cpu_count() or 2)) as ex:
        return list(ex.map(one, zip(items, answers)))

def score(bench, items, answers, tag="run"):
    if bench in ("humaneval", "mbpp"):
        return score_code(bench, items, answers, tag)
    if bench == "bfcl":
        return score_bfcl(items, answers)
    if bench == "livecodebench":
        return score_lcb(items, answers)
    if bench == "ifbench":
        out = []
        for it, a in zip(items, answers):
            r = it["row"]
            inp = ifbench_lib.InputExample(key=r["key"], instruction_id_list=r["instruction_id_list"], prompt=r["prompt"], kwargs=r["kwargs"])
            strict = ifbench_lib.test_instruction_following_strict(inp, {r["prompt"]: a})
            loose = ifbench_lib.test_instruction_following_loose(inp, {r["prompt"]: a})
            out.append({"correct": strict.follow_all_instructions, "loose": loose.follow_all_instructions})
        return out
    if bench == "gsm8k":
        out = []
        for it, a in zip(items, answers):
            pred = extract_final_number(a)
            out.append({"correct": pred is not None and abs(pred - it["gold"]) <= 1e-4 * max(1, abs(it["gold"])), "pred": pred})
        return out
    out = []
    for it, a in zip(items, answers):
        r = it["row"]
        inp = ifeval_lib.InputExample(key=r["key"], instruction_id_list=r["instruction_id_list"], prompt=r["prompt"], kwargs=r["kwargs"])
        strict = ifeval_lib.test_instruction_following_strict(inp, {r["prompt"]: a})
        loose = ifeval_lib.test_instruction_following_loose(inp, {r["prompt"]: a})
        out.append({"correct": strict.follow_all_instructions, "loose": loose.follow_all_instructions})
    return out
print("scorers ready")

## 9. Start the full run on every GPU (in the background)
The next cell writes a small worker script. One copy runs **per GPU**; each loads its own model and:
1. takes questions from a shared list (a question is *claimed* by creating a file, so two GPUs never answer the same one);
2. generates for 64 questions at once, in chunks of ~1,000 tokens; after every chunk, finished answers leave the batch and new questions join, so the GPU stays full instead of waiting for the longest answer;
3. saves every finished answer at once (`answers_gpu*.jsonl`), so after a disconnect *Run All* continues where it stopped.

Questions with the same sampling settings and token cap share a batch; ON and OFF questions can share a batch (only the prompt differs). At start-up each worker also checks that generating in chunks gives the same text as generating in one go.

In [ ]:
%%writefile gen_worker.py
# One of these runs per GPU (started by the next cell).
import argparse, json, os, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ap = argparse.ArgumentParser()
ap.add_argument("--gpu", type=int, required=True)
ap.add_argument("--run_dir", required=True)
ap.add_argument("--model", required=True)
ap.add_argument("--batch", type=int, default=64)
ap.add_argument("--seed", type=int, default=0)
args = ap.parse_args()

def log(*a):
    print(time.strftime("%H:%M:%S"), f"[GPU{args.gpu}]", *a, flush=True)

dev = torch.device(f"cuda:{args.gpu}")
torch.cuda.set_device(dev)
torch.manual_seed(args.seed + args.gpu)
dtype = torch.bfloat16 if torch.cuda.get_device_capability(dev)[0] >= 8 else torch.float16
tok = AutoTokenizer.from_pretrained(args.model, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype, device_map={"": args.gpu},
                                             trust_remote_code=True).eval()
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
_eos = model.generation_config.eos_token_id
EOS = set(_eos if isinstance(_eos, list) else [_eos])
if "<|im_end|>" in tok.get_vocab():
    EOS.add(tok.convert_tokens_to_ids("<|im_end|>"))
EOS.discard(None)
DROP = [t for t in tok.all_special_tokens if t not in ("<think>", "</think>")]
log(f"model loaded ({dtype}), batch size {args.batch}")


@torch.no_grad()
def gen_chunk(seqs, steps, s):
    # seqs: token lists (prompt + answer so far), left-padded into one batch; returns the new tokens per row
    L = max(map(len, seqs))
    ids = torch.full((len(seqs), L), PAD, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, q in enumerate(seqs):
        ids[i, L - len(q):] = torch.tensor(q, dtype=torch.long)
        mask[i, L - len(q):] = 1
    sample = s["temperature"] > 0
    out = model.generate(input_ids=ids.to(dev), attention_mask=mask.to(dev), max_new_tokens=steps,
                         do_sample=sample, temperature=s["temperature"] if sample else None,
                         top_k=s["top_k"] if sample else None, top_p=s["top_p"] if sample else None,
                         repetition_penalty=s["repetition_penalty"],
                         pad_token_id=PAD, eos_token_id=sorted(EOS))
    return out[:, L:].tolist()


def chunk_check():
    # generating 48 tokens in one go must equal 24 + 24 (greedy), also inside a padded batch
    greedy = dict(temperature=0.0, top_k=None, top_p=None, repetition_penalty=1.0)
    enc = lambda q: tok(tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                                add_generation_prompt=True, enable_thinking=False),
                        add_special_tokens=False)["input_ids"]
    a, b = enc("Explain in detail how a hash map works."), enc("Write a long story about the sea. " * 15)
    full = gen_chunk([a, b], 48, greedy)[0]
    p1 = gen_chunk([a, b], 24, greedy)
    chunked = p1[0] + gen_chunk([a + p1[0], b + p1[1]], 24, greedy)[0]
    same = next((k for k, (x, y) in enumerate(zip(full, chunked)) if x != y), min(len(full), len(chunked)))
    log(f"CHECK chunked vs one-go generation: first {same} of {len(full)} tokens identical"
        + (" -> OK" if same >= min(40, len(full)) else " -> WARNING: chunked generation differs early"))


run = args.run_dir
jobs = [json.loads(l) for l in open(os.path.join(run, "jobs.jsonl"))]
done = set()
for f in os.listdir(run):
    if f.startswith("answers_gpu") and f.endswith(".jsonl"):
        for l in open(os.path.join(run, f)):
            try:
                done.add(json.loads(l)["idx"])
            except Exception:
                pass                     # half-written last line after a crash: that question is redone
claims = os.path.join(run, "claims")

def claim(i):
    try:
        os.close(os.open(os.path.join(claims, str(i)), os.O_CREAT | os.O_EXCL))
        return True
    except FileExistsError:
        return False

groups = {}                              # (sampling, token cap) -> question indices still to do
for j in jobs:
    if j["idx"] not in done:
        groups.setdefault((j["sampling_name"], j["max_new_tokens"]), []).append(j["idx"])
order = sorted(groups, key=lambda g: (-g[1], -len(groups[g])))   # longest answers first
out_f = open(os.path.join(run, f"answers_gpu{args.gpu}.jsonl"), "a")

def save(s, truncated):
    j = jobs[s["idx"]]
    text = tok.decode(s["gen"], skip_special_tokens=False)
    for t in DROP:
        text = text.replace(t, "")
    out_f.write(json.dumps({"idx": s["idx"], "key": j["key"], "text": text, "n_tokens": len(s["gen"]),
                            "truncated": truncated, "seconds": s["secs"], "gpu": args.gpu}) + "\n")
    out_f.flush()

chunk_check()
log(f"{len(done)} answers already saved, {sum(map(len, groups.values()))} to go")

cap, active, cur, n_done = args.batch, [], None, 0
while True:
    if not active:                       # batch empty: move to the next group with work left
        cur = next((g for g in order if groups[g]), None)
        if cur is None:
            break
    pend = groups[cur]
    while len(active) < cap and pend:    # refill the batch with new questions
        i = pend.pop(0)
        if claim(i):
            active.append({"idx": i, "gen": [], "secs": 0.0})
    if not active:
        continue
    max_new = cur[1]
    batch = active[:cap]
    remaining = [max_new - len(s["gen"]) for s in batch]
    steps = min(max(256, max_new // 4), max(remaining))
    t0 = time.perf_counter()
    try:
        new = gen_chunk([jobs[s["idx"]]["prompt_ids"] + s["gen"] for s in batch], steps,
                        jobs[batch[0]["idx"]]["sampling"])
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:              # cannot shrink further: count this one as cut off
            s = active.pop(0); save(s, True); log("out of memory on a single question, saved as cut off")
            continue
        cap = max(1, len(batch) // 2)
        log(f"out of memory -> batch size {cap}")
        continue
    secs = time.perf_counter() - t0
    finished = []
    for s, toks, rem in zip(batch, new, remaining):
        toks = toks[:rem]
        stop = next((k for k, t in enumerate(toks) if t in EOS), None)
        s["secs"] += secs / len(batch)
        if stop is not None:
            s["gen"] += toks[:stop]; finished.append((s, False))
        else:
            s["gen"] += toks
            if len(s["gen"]) >= max_new:
                finished.append((s, True))
    for s, truncated in finished:
        active.remove(s); save(s, truncated); n_done += 1
    log(f"{cur[0]}/{cur[1]}: {len(batch)} in batch, {steps} tokens in {secs:.0f}s, "
        f"{len(finished)} finished (this GPU: {n_done})")

out_f.close()
log("all done")

In [ ]:
import hashlib, subprocess, sys

MODES = {"ON": True, "OFF": False}
N_GPUS = torch.cuda.device_count()

# A folder per settings combination, so answers from different settings never mix
_settings = dict(model=MODEL, n=N_PER_BENCH, bfcl=BFCL_PER_CATEGORY, max=MAX_NEW_TOKENS, lcb_max=LCB_MAX_NEW_TOKENS,
                 seed=SEED, sampling=SAMPLING, benchmarks=BENCHMARKS)
RUN_DIR = os.path.join(RESULTS_DIR, "gen_" + hashlib.md5(json.dumps(_settings, sort_keys=True).encode()).hexdigest()[:8])
os.makedirs(os.path.join(RUN_DIR, "claims"), exist_ok=True)

def stop_workers():
    for p in globals().get("WORKERS", []):
        if p.poll() is None:
            p.terminate()
stop_workers()                      # if this cell is run twice, never run two sets of workers

JOBS, PROMPTS = [], {}
for mode, thinking in MODES.items():
    for b in BENCHMARKS:
        for it in ITEMS[b]:
            key = f"{mode}|{b}|{it['id']}"
            PROMPTS[key] = render(it["messages"], thinking, it.get("tools"))
            JOBS.append(dict(idx=len(JOBS), key=key, sampling_name=SAMPLING_OF[b], sampling=SAMPLING[SAMPLING_OF[b]],
                             max_new_tokens=MAX_TOKENS_OF[b],
                             prompt_ids=tokenizer(PROMPTS[key], add_special_tokens=False)["input_ids"]))
with open(os.path.join(RUN_DIR, "jobs.jsonl"), "w") as f:
    f.writelines(json.dumps(j) + "\n" for j in JOBS)
for c in os.listdir(os.path.join(RUN_DIR, "claims")):    # claims left by an interrupted run
    os.remove(os.path.join(RUN_DIR, "claims", c))

WORKERS = []
for g in range(N_GPUS):
    _log = open(os.path.join(RUN_DIR, f"worker_gpu{g}.log"), "a")
    WORKERS.append(subprocess.Popen([sys.executable, os.path.abspath("gen_worker.py"), "--gpu", str(g),
                                     "--run_dir", RUN_DIR, "--model", MODEL, "--batch", str(BATCH_SIZE),
                                     "--seed", str(SEED)], stdout=_log, stderr=subprocess.STDOUT))
try:
    os.nice(10)   # from now on, CPU work in this notebook (scoring) gives way to the GPU workers
except (AttributeError, OSError):
    pass
print(f"Started {N_GPUS} GPU worker(s) for {len(JOBS)} answers -> {RUN_DIR}")
print("They run in the background. The next sections continue meanwhile; section 12 collects the answers.")

## 10. Scorer self-test (no model needed, runs while the GPUs work)
Before trusting any number we prove the scoring is right: **reference solutions must score ~100%**, **wrong answers 0%**.
- IFEval: we replay GPT-4's published responses, which should score about **77%** (the paper reports 76.9%).
- IFBench: we replay AllenAI's sample outputs and must **agree with their official per-prompt results**.
- BFCL: reference tool calls are written in LFM3's XML format from the answer key, then parsed back and checked — this tests our parser too.
- LiveCodeBench: two hand-checked solutions must pass all hidden tests.

In [ ]:
_gpt4_path = os.path.join(DATA_DIR, "ifeval_gpt4.jsonl")
if not os.path.exists(_gpt4_path):
    download("https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/data/input_response_data_gpt4_20231107_145030.jsonl", _gpt4_path)
GPT4 = {r["prompt"]: r["response"] for r in read_jsonl("ifeval_gpt4.jsonl")}
HE_BY_ID, MBPP_BY_ID = {r["task_id"]: r for r in HE_ROWS}, {r["task_id"]: r for r in MBPP_ROWS}

def reference_answer(it, good=True):
    if it["bench"] == "humaneval":
        r = HE_BY_ID[it["id"]]
        return (f"```python\n{r['prompt']}{r['canonical_solution']}```" if good
                else f"```python\ndef {r['entry_point']}(*a, **k):\n    return None\n```")
    if it["bench"] == "mbpp":
        return f"```python\n{MBPP_BY_ID[it['id']]['canonical_solution']}\n```" if good else "```python\nprint('hi')\n```"
    if it["bench"] == "gsm8k":
        g = it["gold"]; return f"\\boxed{{{int(g) if g == int(g) else g}}}" if good else "\\boxed{-999999}"
    if it["bench"] == "bfcl":
        return bfcl_reference(it) if good else (
            "<tool_call>\n<function=not_a_real_tool>\n</function>\n</tool_call>" if it["category"] == "irrelevance"
            else "I am not sure which tool to use.")
    if it["bench"] == "ifbench":
        return IFB_SAMPLE.get(it["row"]["prompt"], "") if good else ""
    if it["bench"] == "livecodebench":
        return LCB_REFERENCE.get(it["id"], "") if good else "```python\nprint(0)\n```"
    return GPT4.get(it["row"]["prompt"], "") if good else ""

# BFCL: write the answer key as LFM3 tool calls (strings raw, everything else JSON — like the chat template)
def _first(options):
    return next((o for o in options if o != ""), None)
def _pick(v):   # answer keys list the allowed options for every (nested) value; take the first real one
    if isinstance(v, dict):
        return {k: _pick(_first(o)) for k, o in v.items() if _first(o) is not None}
    if isinstance(v, list) and v and all(isinstance(x, dict) for x in v):
        return [_pick(x) for x in v]
    return v
def _render_value(v):
    if isinstance(v, str): return v
    if isinstance(v, bool): return "true" if v else "false"
    if v is None: return "null"
    return json.dumps(v) if isinstance(v, (list, dict)) else str(v)
def bfcl_reference(it):
    if it["category"] == "irrelevance":
        return "None of the available tools can answer this, so I won't call any."
    blocks = []
    for call in it["ground_truth"]:
        for fname, params in call.items():
            args = "".join(f"<parameter={k}>\n{_render_value(_pick(_first(o)))}\n</parameter>\n"
                           for k, o in params.items() if _first(o) is not None)
            blocks.append(f"<tool_call>\n<function={fname}>\n{args}</function>\n</tool_call>")
    return "\n\n".join(blocks)

# IFBench: AllenAI's sample outputs and their official strict results
IFB_SAMPLE = {r["prompt"]: r["response"] for r in read_jsonl("ifbench_sample_output.jsonl")}
IFB_OFFICIAL = {r["prompt"]: r["follow_all_instructions"] for r in read_jsonl("ifbench_official_strict.jsonl")}

# LiveCodeBench: two hand-written, hand-checked solutions (one stdin problem, one LeetCode-style function)
LCB_REFERENCE = {
    "lcb/abc387_b": "```python\nX = int(input())\nprint(sum(i * j for i in range(1, 10) for j in range(1, 10) if i * j != X))\n```",
    "lcb/3708": ("```python\nclass Solution:\n    def zigzagTraversal(self, grid: List[List[int]]) -> List[int]:\n"
                 "        flat = []\n        for i, row in enumerate(grid):\n            flat += row if i % 2 == 0 else row[::-1]\n"
                 "        return flat[::2]\n```"),
}
_lcb_by_id = {f"lcb/{r['question_id']}": r for r in LCB_ROWS}
LCB_SELFTEST = [dict(bench="livecodebench", id=k, row=_lcb_by_id[k], question="") for k in LCB_REFERENCE]

def run_selftest():
    print(f"{'benchmark':14s} {'reference':>10s} {'wrong':>7s}")
    for b in BENCHMARKS:
        its = LCB_SELFTEST if b == "livecodebench" else ITEMS[b]
        good_scores = score(b, its, [reference_answer(i, True) for i in its], "selftest_good")
        good = sum(s["correct"] for s in good_scores) / len(its)
        bad = sum(s["correct"] for s in score(b, its, [reference_answer(i, False) for i in its], "selftest_bad")) / len(its)
        extra = ""
        if b == "ifbench":   # must match AllenAI's official verdicts prompt by prompt
            pairs = [(s["correct"], IFB_OFFICIAL[i["row"]["prompt"]]) for i, s in zip(its, good_scores) if i["row"]["prompt"] in IFB_OFFICIAL]
            agree = sum(a == o for a, o in pairs) / max(1, len(pairs))
            extra = f"   agreement with official results: {agree:.1%} ({len(pairs)} prompts)"
            assert agree >= 0.95, "IFBench scorer disagrees with the official results"
        elif b == "bfcl":
            failed = [i["id"] for i, s in zip(its, good_scores) if not s["correct"]]
            extra = f"   failed references: {failed}" if failed else ""
        print(f"{b:14s} {good:10.1%} {bad:7.1%}{extra}", flush=True)
        assert bad == 0, f"{b}: wrong answers should score 0"
        if b != "ifbench":
            assert good >= {"ifeval": 0.5, "bfcl": 0.9}.get(b, 0.95), f"{b}: reference answers scored too low"

try:
    run_selftest()
except Exception:
    if "stop_workers" in globals():
        stop_workers(); print("Scorer self-test FAILED, so the GPU workers were stopped.")
    raise
print("\nScorer self-test passed.")

## 11. First look: one question, both modes
Same question, thinking ON vs OFF, side by side (click "show thinking" to see the reasoning). This runs on GPU 0 next to its worker, so it is a bit slower than usual.

In [ ]:
def ask(question, thinking, sampling="general", max_new_tokens=MAX_NEW_TOKENS, tools=None):
    # question: a string, or a full list of chat messages
    msgs = question if isinstance(question, list) else [{"role": "user", "content": question}]
    p = render(msgs, thinking, tools)
    g = generate([p], SAMPLING[sampling], max_new_tokens=max_new_tokens, batch_size=1, label="ON" if thinking else "OFF")[0]
    reasoning, answer, status = split_thinking(g["text"], prompt_opens_thinking(p))
    return dict(reasoning=reasoning, answer=answer, status=status, **g)

def compare(question, sampling="general"):
    on, off = ask(question, True, sampling), ask(question, False, sampling)
    display(HTML(f"<div style='background:#000;padding:16px;border-radius:12px;font-family:sans-serif'>"
                 f"<div style='background:#e0e0e0;color:#000;padding:10px;border-radius:8px;margin-bottom:12px'>"
                 f"<b>QUESTION:</b> {html.escape(question)}</div><div style='display:flex;gap:12px'>"
                 f"{_panel('THINKING ON', '#e67e22', on)}{_panel('THINKING OFF', '#007bff', off)}</div></div>"))
    return on, off

_ = compare("A shop sells pens at 3 for $4. How much do 18 pens cost? Put the final answer in \\boxed{}.", "math")

## 12. Collect the answers and score them (while the GPUs keep working)
This cell waits for the GPU workers. As soon as all answers of one benchmark + mode are in, it scores them and prints the result, so you see numbers long before the end. The status line shows progress and a rough time left (rough because thinking-ON answers, which are longer, go first).

If Kaggle disconnects: **Run All** again. Saved answers are kept and only the missing ones are generated.

In [ ]:
def read_answers():
    got = {}
    for f in sorted(os.listdir(RUN_DIR)):
        if f.startswith("answers_gpu") and f.endswith(".jsonl"):
            for l in open(os.path.join(RUN_DIR, f)):
                try:
                    r = json.loads(l)
                except ValueError:
                    continue             # line still being written
                got.setdefault(r["key"], r)
    return got

def worker_log(g, last=3000):
    p = os.path.join(RUN_DIR, f"worker_gpu{g}.log")
    return open(p).read()[-last:] if os.path.exists(p) else ""

def score_group(mode, b, got):
    its, thinking, rows = ITEMS[b], MODES[mode], []
    for it in its:
        key = f"{mode}|{b}|{it['id']}"
        g = got[key]
        reasoning, answer, status = split_thinking(g["text"], prompt_opens_thinking(PROMPTS[key]))
        rows.append(dict(mode=mode, bench=b, id=it["id"], question=it["question"], raw=g["text"],
                         reasoning=reasoning, answer=answer, think_status=status,
                         n_tokens=g["n_tokens"], reasoning_tokens=count_tokens(reasoning),
                         truncated=g["truncated"], seconds=g["seconds"], gpu=g["gpu"],
                         leak=(not thinking) and leaked_thinking(g["text"])))
    for r, s in zip(rows, score(b, its, [r["answer"] for r in rows], tag=f"{mode}_{b}")):
        r.update(correct=bool(s["correct"]))
    return rows

GROUPS = {(m, b): [f"{m}|{b}|{it['id']}" for it in ITEMS[b]] for m in MODES for b in BENCHMARKS}
SCORED, _checks_shown = {}, False
_status = display(HTML("<pre>waiting for the GPU workers...</pre>"), display_id=True)
_t0, _n0 = time.time(), len(read_answers())

while True:
    got = read_answers()
    if not _checks_shown and all("CHECK" in worker_log(g, 100000) for g in range(len(WORKERS))):
        for g in range(len(WORKERS)):
            print("\n".join(l for l in worker_log(g, 100000).splitlines() if "CHECK" in l))
        _checks_shown = True
    for (m, b), keys in GROUPS.items():
        if (m, b) not in SCORED and all(k in got for k in keys):
            rows = SCORED[(m, b)] = score_group(m, b, got)
            print(f"{b:14s} thinking {m:3s}: accuracy {sum(r['correct'] for r in rows)/len(rows):6.1%} | "
                  f"mean tokens {sum(r['n_tokens'] for r in rows)/len(rows):5.0f} | "
                  f"cut off {sum(r['truncated'] for r in rows)/len(rows):4.0%}", flush=True)
    alive = sum(p.poll() is None for p in WORKERS)
    mins = (time.time() - _t0) / 60
    rate = (len(got) - _n0) / max(mins, 1e-9)
    left = f"{(len(JOBS) - len(got)) / rate:.0f} min" if rate > 0 else "?"
    _status.update(HTML(f"<pre>{len(got)}/{len(JOBS)} answers | {alive}/{len(WORKERS)} GPU workers running | "
                        f"{mins:.0f} min so far | rough time left: {left} | scored {len(SCORED)}/{len(GROUPS)}</pre>"))
    if len(SCORED) == len(GROUPS):
        break
    if alive == 0 and len(read_answers()) < len(JOBS):
        for g in range(len(WORKERS)):
            print(f"----- worker_gpu{g}.log (end) -----\n{worker_log(g)}")
        raise RuntimeError("The GPU workers stopped before finishing (see the logs above). "
                           "Re-run the cell in section 9 and then this one: they continue where they stopped.")
    time.sleep(20)

RECORDS = [r for m in MODES for b in BENCHMARKS for r in SCORED[(m, b)]]
with open(os.path.join(RESULTS_DIR, "all_answers.jsonl"), "w") as f:
    f.writelines(json.dumps(r) + "\n" for r in RECORDS)
print(f"\nDone: {len(RECORDS)} answers scored in {(time.time() - _t0) / 60:.0f} min.")

## 13. Results
`sec_per_answer` is GPU time shared out over the answers generated together (64 at once), not the time a user waits; section 16 measures that.

In [ ]:
import pandas as pd
df = pd.DataFrame(RECORDS)

summary = (df.groupby(["bench", "mode"], sort=False)
             .agg(n=("correct", "size"), accuracy=("correct", "mean"),
                  mean_tokens=("n_tokens", "mean"), median_tokens=("n_tokens", "median"),
                  reasoning_tokens=("reasoning_tokens", "mean"), truncated=("truncated", "mean"),
                  leaked_thinking=("leak", "mean"), sec_per_answer=("seconds", "mean"))
             .reset_index())
display(summary.style.format({"accuracy": "{:.1%}", "truncated": "{:.1%}", "leaked_thinking": "{:.1%}",
                              "mean_tokens": "{:.0f}", "median_tokens": "{:.0f}", "reasoning_tokens": "{:.0f}",
                              "sec_per_answer": "{:.1f}"}))

on = summary[summary["mode"] == "ON"].set_index("bench"); off = summary[summary["mode"] == "OFF"].set_index("bench")
gap = pd.DataFrame({"accuracy ON": on["accuracy"], "accuracy OFF": off["accuracy"],
                    "gap (points)": (on["accuracy"] - off["accuracy"]) * 100,
                    "OFF keeps % of ON": off["accuracy"] / on["accuracy"].replace(0, float("nan")) * 100,
                    "tokens ON": on["mean_tokens"], "tokens OFF": off["mean_tokens"],
                    "OFF uses x fewer tokens": on["mean_tokens"] / off["mean_tokens"].replace(0, float("nan"))})
gap.loc["AVERAGE"] = gap.mean(numeric_only=True)
print("\nThinking ON vs OFF")
display(gap.style.format({"accuracy ON": "{:.1%}", "accuracy OFF": "{:.1%}", "gap (points)": "{:+.1f}",
                          "OFF keeps % of ON": "{:.0f}%", "tokens ON": "{:.0f}", "tokens OFF": "{:.0f}",
                          "OFF uses x fewer tokens": "{:.1f}x"}))

# BFCL by category (irrelevance = correctly NOT calling a tool)
if "bfcl" in BENCHMARKS:
    _cat = {it["id"]: it["category"] for it in ITEMS["bfcl"]}
    bf = df[df.bench == "bfcl"].assign(category=lambda d: d["id"].map(_cat))
    print("\nBFCL accuracy by category")
    display(bf.pivot_table(index="category", columns="mode", values="correct", aggfunc="mean")
              .reindex(BFCL_CATEGORIES)[["ON", "OFF"]].style.format("{:.0%}"))

# Sanity check against Liquid's own numbers (thinking ON). Their setup differs: temperature 0.6,
# no output limit, all problems, full BFCL V4 (incl. agentic parts) — so expect our ON a bit lower.
LIQUID = {"livecodebench": 73.46, "ifeval": 96.66, "ifbench": 73.03, "bfcl": 71.39}
cmp_ = pd.DataFrame({"Liquid reported (ON)": pd.Series(LIQUID) / 100,
                     "ours ON": on["accuracy"], "ours OFF": off["accuracy"]}).dropna(subset=["ours ON"])
print("\nOur thinking-ON scores vs Liquid's published numbers (different settings, see comment)")
display(cmp_.style.format("{:.1%}", na_rep="–"))

summary.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
gap.to_csv(os.path.join(RESULTS_DIR, "on_vs_off.csv"))
with open(os.path.join(RESULTS_DIR, "summary.md"), "w") as f:
    f.write(f"# {MODEL}: thinking ON vs OFF (questions: {', '.join(f'{b} {len(ITEMS[b])}' for b in BENCHMARKS)}; max_new_tokens={MAX_NEW_TOKENS}, seed={SEED})\n\n")
    f.write(summary.to_markdown(index=False, floatfmt=".3f") + "\n\n" + gap.to_markdown(floatfmt=".2f") + "\n")

## 14. Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(BENCHMARKS)); w = 0.38
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for k, (col, title, fmt) in enumerate([("accuracy", "Accuracy (higher is better)", "{:.0%}"),
                                        ("mean_tokens", "Mean tokens per answer (lower is cheaper)", "{:.0f}")]):
    for j, (mode, color) in enumerate([("ON", "#e67e22"), ("OFF", "#007bff")]):
        vals = [summary[(summary.bench == b) & (summary["mode"] == mode)][col].iloc[0] for b in BENCHMARKS]
        bars = ax[k].bar(x + (j - 0.5) * w, vals, w, label=f"thinking {mode}", color=color)
        ax[k].bar_label(bars, labels=[fmt.format(v) for v in vals], fontsize=8)
    ax[k].set_xticks(x, BENCHMARKS); ax[k].set_title(title); ax[k].legend()
    ax[k].spines[["top", "right"]].set_visible(False)
ax[0].set_ylim(0, 1.1)
plt.suptitle(f"{MODEL}: thinking ON vs OFF"); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "on_vs_off.png"), dpi=150); plt.show()

## 15. Where does thinking OFF fail?
For each question: did both modes get it right, only ON, only OFF, or neither? The **"only ON right"** questions are exactly what fine-tuning must fix — they show what the model loses without thinking.

In [ ]:
wide = df.pivot_table(index=["bench", "id"], columns="mode", values="correct", aggfunc="first").reset_index()
_on, _off = wide["ON"].astype(bool), wide["OFF"].astype(bool)
wide["outcome"] = np.select([_on & _off, _on & ~_off, ~_on & _off],
                            ["both right", "only ON right", "only OFF right"], "both wrong")
display(pd.crosstab(wide["bench"], wide["outcome"], margins=True))

lost = wide[wide.outcome == "only ON right"]
print(f"\n{len(lost)} questions solved with thinking but failed without it. Two examples:")
by_key = {(r["mode"], r["id"]): r for r in RECORDS}
for _, row in lost.head(2).iterrows():
    a_on, a_off = by_key[("ON", row["id"])], by_key[("OFF", row["id"])]
    display(HTML(f"<div style='background:#000;padding:14px;border-radius:12px;font-family:sans-serif;margin:8px 0'>"
                 f"<div style='background:#e0e0e0;color:#000;padding:8px;border-radius:8px;margin-bottom:10px'>"
                 f"<b>{row['bench']} {row['id']}</b><pre style='white-space:pre-wrap'>{html.escape(a_on['question'][:1200])}</pre></div>"
                 f"<div style='display:flex;gap:12px'>{_panel('ON (correct)', '#2e7d32', a_on)}{_panel('OFF (wrong)', '#c62828', a_off)}</div></div>"))
wide.to_csv(os.path.join(RESULTS_DIR, "per_question_outcomes.csv"), index=False)

## 16. Speed test: how long does a user wait?
One question at a time (batch size 1), like a real chat on a laptop or phone. This is the number users feel.

In [ ]:
speed = []
for b in BENCHMARKS:
    it = ITEMS[b][0]
    for mode, thinking in MODES.items():
        g = ask(it["messages"], thinking, SAMPLING_OF[b], MAX_TOKENS_OF[b], it.get("tools"))
        speed.append(dict(bench=b, mode=mode, tokens=g["n_tokens"], seconds=g["seconds"],
                          tokens_per_sec=g["n_tokens"] / max(g["seconds"], 1e-9)))
speed = pd.DataFrame(speed)
display(speed.pivot(index="bench", columns="mode", values=["seconds", "tokens", "tokens_per_sec"]).round(1))
t_on, t_off = speed[speed["mode"] == "ON"].seconds.sum(), speed[speed["mode"] == "OFF"].seconds.sum()
print(f"Waiting time for these {len(BENCHMARKS)} questions: ON {t_on:.0f}s vs OFF {t_off:.0f}s  "
      f"({t_on / max(t_off, 1e-9):.1f}x faster without thinking)")
speed.to_csv(os.path.join(RESULTS_DIR, "speed_test.csv"), index=False)

## 17. Interactive sandbox
Ask anything and flip the thinking switch.

In [ ]:
import ipywidgets as widgets

box = widgets.Text(placeholder="Ask LFM3 something...", description="Message:", layout=widgets.Layout(width="70%"))
think = widgets.Checkbox(value=False, description="Thinking ON")
both = widgets.Checkbox(value=False, description="Compare both")
out = widgets.Output()

def _on_submit(_):
    q = box.value.strip()
    if not q: return
    with out:
        if both.value:
            compare(q)
        else:
            g = ask(q, think.value)
            display(HTML(f"<div style='display:flex'>{_panel('THINKING ' + ('ON' if think.value else 'OFF'), '#e67e22' if think.value else '#007bff', g)}</div>"))
    box.value = ""

box.on_submit(_on_submit)
display(widgets.HBox([box, think, both]), out)

## 18. Save results
Everything is in `results/` and zipped to `results.zip` (Output panel → download). Send `summary.md`, `on_vs_off.png` and `per_question_outcomes.csv` for review.

In [ ]:
shutil.make_archive(os.path.join(WORK, "results"), "zip", RESULTS_DIR)
print("saved", os.path.join(WORK, "results.zip"))
print(open(os.path.join(RESULTS_DIR, "summary.md")).read())